In [1]:
import os

import numpy as np
import pandas as pd
import pytorch_lightning as pl
import scanpy as sc
import torch

import T_perturb
from T_perturb.Perturb.datamodule import PerturberDataModule
from T_perturb.Perturb.trainer import PerturberTrainer

from T_perturb.src.utils import label_encoder
from T_perturb.tests.test_cellgen_training import dummy_dataset
from T_perturb.tests.test_countdecoder_training import dummy_cell_gene_matrix

/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
seed_no = 42
pl.seed_everything(seed_no)
torch.manual_seed(seed_no)

Seed set to 42


In [3]:
if os.getcwd().split('/')[-1] != 'healthy_imm_expr':
    # set working directory to root of repository
    os.chdir('/lustre/scratch126/cellgen/team361/kl11/t_generative/')

In [4]:
tgt_vocab_size = 101  # +1 for padding token
num_samples = 100
num_genes = 100
max_seq_length = 50
n_total_tps = 2
num_samples = 100
batch_size = 4
pred_tps = [1, 2]
context_tps = [1, 2]
d_model = 12

genes_to_perturb = [70]
perturbation_token = 1


In [5]:
src_counts = dummy_cell_gene_matrix(
    num_cells=num_samples,
    num_genes=num_genes,
)
src_dataset = dummy_dataset(
    max_len=max_seq_length,
    vocab_size=tgt_vocab_size,
    num_samples=100,
)
tgt_counts_dict = dummy_cell_gene_matrix(
    num_cells=num_samples,
    num_genes=num_genes,
    total_time_steps=n_total_tps,
)
tgt_datasets = dummy_dataset(
    max_len=max_seq_length,
    vocab_size=tgt_vocab_size,
    num_samples=100,
    total_time_steps=n_total_tps,
)

In [6]:
# check if cuda is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [9]:
import importlib
import T_perturb.src.utils
importlib.reload(T_perturb.Perturb.trainer)
importlib.reload(T_perturb.Perturb.datamodule)
importlib.reload(T_perturb.src.utils)
from T_perturb.Perturb.trainer import PerturberTrainer
from T_perturb.Perturb.datamodule import PerturberDataModule

trainer_params = {
    'tgt_vocab_size': tgt_vocab_size,
    'd_model': d_model,
    'num_heads': 4,
    'num_layers': 1,
    'd_ff': 8,
    'max_seq_length': max_seq_length + 10,
    'dropout': 0.0,
    'pred_tps': pred_tps,
    'context_tps': context_tps,
    'n_total_tps': n_total_tps,
    'precision': 'high',
    'mask_scheduler': 'pow',
    'output_dir': './T_perturb/T_perturb/tests/res',
    'encoder': 'Transformer_encoder',
    'var_list': None,
    'genes_to_perturb': genes_to_perturb,
    'perturbation_token': perturbation_token,
    'context_mode': True,
    'perturbation_mode': 'generate',
    'perturbation_sequence': ['src','tgt'],
    'temperature':1.5,
    'iterations': 19,
    'sequence_length': max_seq_length - 10,
    'pos_encoding_mode': 'time_pos_sin',
    'return_attn': False,
    'mask_scheduler': 'cosine'
}
decoder_module = PerturberTrainer(
    **trainer_params
)

celltype_to_perturb = 'A'
data_module = PerturberDataModule(
    src_counts=src_counts,
    tgt_counts_dict=tgt_counts_dict,
    src_dataset=src_dataset,
    tgt_datasets=tgt_datasets,
    batch_size=batch_size,
    num_workers=1,
    pred_tps=pred_tps,
    context_tps=context_tps,
    n_total_tps=n_total_tps,
    split=False,
    max_len=max_seq_length,
    celltype_to_perturb=celltype_to_perturb,
    filter_mode='include'
)
data_module.setup()
test_loader = data_module.test_dataloader()

cls_token_1 tensor([101])
cls_token_2 tensor([102])
mapping dict None
Start datamodule
src_len 32
tgt_len 43


/lustre/scratch126/cellgen/team361/kl11/t_generative/T_perturb/T_perturb/Dataloaders/datamodule.py:62: UserWarning: src and tgt dataset have different length
  warn('src and tgt dataset have different length')


In [10]:
accelerator = 'gpu' if torch.cuda.is_available() else 'cpu'
trainer = pl.Trainer(
    limit_test_batches=1,  # Limit to a single batch for quick testing
    logger=False,
    accelerator=accelerator,
    devices=1 if torch.cuda.is_available() else 0,  # inference only on one gpu
)
trainer.test(
    decoder_module, 
    data_module,
    ckpt_path='T_perturb/T_perturb/tests/checkpoints/test_masking_checkpoint-epoch=00.ckpt',
    )


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
`Trainer(limit_test_batches=1)` was configured so 1 batch will be used.
Restoring states from the checkpoint path at T_perturb/T_perturb/tests/checkpoints/test_masking_checkpoint-epoch=00.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at T_perturb/T_perturb/tests/checkpoints/test_masking_checkpoint-epoch=00.ckpt


src_len 32
tgt_len 43


/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:424: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Testing DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 19/19 [00:00<00:00, 103.48it/s]


Testing DataLoader 0: 100%|██████████| 1/1 [00:00<00:00,  1.06it/s]

/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


---Generating anndata
anndata generation completed---
Testing DataLoader 0: 100%|██████████| 1/1 [00:00<00:00,  1.02it/s]


[{}]